In [29]:
%pip install agentpy
%pip install numphy
%pip install random

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
ERROR: Could not find a version that satisfies the requirement random (from versions: none)
ERROR: No matching distribution found for random
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [30]:
import numpy as np
import agentpy as ap
import random


In [31]:
environmentRows = 11
environmentColumns = 11
field = np.full((environmentRows, environmentColumns), 1)

q_values = np.zeros((environmentRows, environmentColumns, 4))

actions = ['up', 'right', 'down', 'left']

In [32]:
def getIndicesToChange():
  indices_to_change = set()
  while len(indices_to_change) < 4:
      row = random.randint(0, environmentRows - 1)
      col = random.randint(0, environmentColumns - 1)
      indices_to_change.add((row, col))
  return indices_to_change

def createObstacles(matrix, indices_to_change):
    # Change the values at the selected indices to -100
    for row, col in indices_to_change:
        matrix[row][col] = -100

    return matrix

def fieldIsEmpty(matrix):

    for row in matrix:
        for element in row:
            if element == 1:
                return False
    return True


#define a function that determines if the specified location is a terminal state
def is_terminal_state(current_row_index, current_column_index):
  #if the reward for this location is -1, then it is not a terminal state (i.e., it is a 'white square')
  if fieldIsEmpty(field): 
    return True
  
  if (field[current_row_index, current_column_index] == -1 or field[current_row_index, current_column_index] == 1):
    return False

  return True




#define an epsilon greedy algorithm that will choose which action to take next (i.e., where to move next)
def get_next_action(current_row_index, current_column_index, epsilon):
  #if a randomly chosen value between 0 and 1 is less than epsilon,
  #then choose the most promising value from the Q-table for this state.
  if np.random.random() < epsilon:
    return np.argmax(q_values[current_row_index, current_column_index])
  else: #choose a random action
    return np.random.randint(4)

#define a function that will get the next location based on the chosen action
def get_next_location(current_row_index, current_column_index, action_index):
  new_row_index = current_row_index
  new_column_index = current_column_index
  if actions[action_index] == 'up' and current_row_index > 0:
    new_row_index -= 1
  elif actions[action_index] == 'right' and current_column_index < environmentColumns - 1:
    new_column_index += 1
  elif actions[action_index] == 'down' and current_row_index < environmentRows - 1:
    new_row_index += 1
  elif actions[action_index] == 'left' and current_column_index > 0:
    new_column_index -= 1
  return new_row_index, new_column_index


#Define a function that will get the shortest path between any location within the warehouse that
#the robot is allowed to travel and the item packaging location.
def get_shortest_path(start_row_index, start_column_index):
  #return immediately if this is an invalid starting location
  if is_terminal_state(start_row_index, start_column_index):
    return []
  else: #if this is a 'legal' starting location
    current_row_index, current_column_index = start_row_index, start_column_index
    shortest_path = []
    shortest_path.append([current_row_index, current_column_index])
    #continue moving along the path until we reach the goal (i.e., the item packaging location)
    while not is_terminal_state(current_row_index, current_column_index):
      #get the best action to take
      action_index = get_next_action(current_row_index, current_column_index, 1.)
      #move to the next location on the path, and add the new location to the list
      current_row_index, current_column_index = get_next_location(current_row_index, current_column_index, action_index)
      shortest_path.append([current_row_index, current_column_index])
    return shortest_path

In [33]:
# Generate a list of unique indices to change
indicesToChange = getIndicesToChange()
createObstacles(field, indicesToChange)

for row in field:
  print(row)
#define training parameters
epsilon = 0.9 #the percentage of time when we should take the best action (instead of a random action)
discount_factor = 0.9 #discount factor for future rewards
learning_rate = 0.9 #the rate at which the AI agent should learn

dic = {}
#run through 1000 training episodes

for episode in range(10000):
  #get the starting location for this episode
  row_index, column_index = 0, 0

  field = np.full((environmentRows, environmentColumns), 1)

  createObstacles(field, indicesToChange)
  episodeDic = {}

  #continue taking actions (i.e., moving) until we reach a terminal state
  #(i.e., until we reach the item packaging area or crash into an item storage location)
  cont = 0
  success = False
  while not is_terminal_state(row_index, column_index) :
    #choose which action to take (i.e., where to move next)
    action_index = get_next_action(row_index, column_index, epsilon)

    #perform the chosen action, and transition to the next state (i.e., move to the next location)
    old_row_index, old_column_index = row_index, column_index #store the old row and column indexes
    row_index, column_index = get_next_location(row_index, column_index, action_index)

    #receive the reward for moving to the new state, and calculate the temporal difference
    reward = field[row_index, column_index]
   
    if reward == 1: field[row_index, column_index] = -1
    if fieldIsEmpty(field): 
      reward = 10000000000000
      success = True

    old_q_value = q_values[old_row_index, old_column_index, action_index]
    temporal_difference = reward + (discount_factor * np.max(q_values[row_index, column_index])) - old_q_value

    #update the Q-value for the previous state and action pair
    new_q_value = old_q_value + (learning_rate * temporal_difference)
    q_values[old_row_index, old_column_index, action_index] = new_q_value
    cont += 1

    episodeDic[f"step{cont}"] = {
                # 'harvesterPosition': self.position,
                # 'tractorPosition': self.tractor.position,
                # 'containerPosition': self.tractor.container.position,
                # 'containerLoad': self.tractor.container.load,
                "field": [row[:] for row in field],
            }
  
  dic[f"episode{episode}"] = {
    "episodes": episodeDic,
    "success": success
    }
  if (success): print(cont)
print('Training complete!')

[1 1 1 1 1 1 1 1 1 1 1]
[   1    1    1    1    1    1    1    1 -100    1    1]
[   1    1    1    1    1    1    1    1    1    1 -100]
[1 1 1 1 1 1 1 1 1 1 1]
[1 1 1 1 1 1 1 1 1 1 1]
[   1    1    1    1    1    1    1    1    1 -100 -100]
[1 1 1 1 1 1 1 1 1 1 1]
[1 1 1 1 1 1 1 1 1 1 1]
[1 1 1 1 1 1 1 1 1 1 1]
[1 1 1 1 1 1 1 1 1 1 1]
[1 1 1 1 1 1 1 1 1 1 1]
1194
1670
1128
9591
1522
1370
1464
893
1606
1277
927
756
877
842
873
619
5732
1063
728
554
15562
643
725
2292
721
997
905
11990
958
1322
9498
806
1060
4736
3681
2871
6209
827
846
6014
657
779
744
800
1101
1026
886
11362
710
896
890
757
599
1493
892
1227
915
1439
1055
605
1096
1210
973
1258
1982
2910
791
695
13989
867
547
1408
732
875
539
11761
704
1040
2245
1051
2312
915
1004
1743
877
897
16436
987
840
659
13327
838
676
954
1844
1197
846
710
671
901
8126
991
964
1146
922
1060
1375
643
551
966
784
870
910
3919
879
1011
1011
783
867
1399
923
675
984
4563
1313
6054
1134
859
1168
1099
Training complete!


In [34]:
print(get_shortest_path(0,0))